In [ ]:
import pandas as pd
import plotly.graph_objects as go

In [ ]:
!pip install kaleido==0.2.1 -q

In [ ]:
df = pd.read_csv("df_raw.csv", parse_dates=["timestamp"])

print(f"Строк: {len(df)}")
print(f"Период: {df['timestamp'].min()} - {df['timestamp'].max()}")

In [ ]:
houses = ["house_1", "house_2", "house_3", "house_4", "house_5", "house_6", "house_7", "house_8"]

colors = {
    "house_1": "#1f77b4", "house_2": "#ff7f0e", "house_3": "#2ca02c",
    "house_4": "#d62728", "house_5": "#9467bd", "house_6": "#8c564b",
    "house_7": "#e377c2", "house_8": "#7f7f7f"
}

fig = go.Figure()

for house in houses:
    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=df[house],
        mode="lines",
        name=house,
        line=dict(width=0.7, color=colors[house]),
    ))

fig.update_layout(
    title="Сырые данные по всем домам",
    xaxis_title="Дата",
    yaxis_title="Мощность, кВт",
    width=700,
    height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified",
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("01_raw_data_all_houses.png", scale=2)

In [ ]:
# скользящее среднее 7 дней (7 * 48 = 336 интервалов по 30 минут)
window = 7 * 48

fig = go.Figure()

for house in houses:
    rolling = df[house].rolling(window=window, center=True).mean()

    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=rolling,
        mode="lines",
        name=house,
        line=dict(width=1.5, color=colors[house]),
    ))

fig.update_layout(
    title="Скользящее среднее 7 дней по всем домам",
    xaxis_title="Дата",
    yaxis_title="Мощность, кВт",
    width=700,
    height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified",
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("02_rolling_mean_all_houses.png", scale=2)

In [ ]:
# Для дома №3
mask = (
    (df["timestamp"] >= "2018-01-01") &
    (df["timestamp"] <= "2019-06-01")
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df[mask]["timestamp"],
    y=df[mask]["house_3"],
    mode="lines",
    line=dict(width=1, color="#2ca02c"),
    name="house_3"
))
fig.update_layout(
    title="Дом 3: поиск провала значений мощностей",
    xaxis_title="Дата",
    yaxis_title="Мощность, кВт",
    width=700,
    height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified"
)
fig.show()
fig.write_image("03_house3_problem.png.png", scale=2)

In [ ]:
# Для дома №5
mask = (
    (df["timestamp"] >= "2019-08-01") &
    (df["timestamp"] <= "2019-12-06")
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df[mask]["timestamp"],
    y=df[mask]["house_5"],
    mode="lines",
    line=dict(width=1, color="#9467bd"),
    name="house_5"
))
fig.update_layout(
    title="Дом 5: август 2019 - декабрь 2019",
    xaxis_title="Дата",
    yaxis_title="Мощность, кВт",
    width=700,
    height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified"
)
fig.show()
fig.write_image("05_house5_problem.png", scale=2)

In [ ]:
# Для дома №6
mask = (
    (df["timestamp"] >= "2017-09-01") &
    (df["timestamp"] <= "2018-03-01")
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df[mask]["timestamp"],
    y=df[mask]["house_6"],
    mode="lines",
    line=dict(width=1, color="#8c564b"),
    name="house_6"
))
fig.update_layout(
    title="Дом 6: сентябрь 2017 - март 2018",
    xaxis_title="Дата",
    yaxis_title="Мощность, кВт",
    width=700,
    height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified"
)
fig.show()
fig.write_image("04_house6_problem.png", scale=2)

In [ ]:
for col in ["house_1", "house_2", "house_3", "house_4", "house_5", "house_6", "house_7"]:
    missing = df[df[col].isna()]["timestamp"]
    if len(missing) == 0:
        continue

    gaps = []
    start = missing.iloc[0]
    prev = missing.iloc[0]

    for ts in missing.iloc[1:]:
        if ts - prev > pd.Timedelta("30min"):
            gaps.append((start, prev))
            start = ts
        prev = ts
    gaps.append((start, prev))

    print(f"{col} - блоков пропусков: {len(gaps)}")
    for g_start, g_end in gaps:
        duration = (g_end - g_start).total_seconds() / 3600
        print(f"{g_start} - {g_end} ({round(duration)} ч)")
    print()

In [ ]:
# заполнение пропусков данными того же периода ±364 дня
fill_gaps = [
    # дом 3 - провал сентябрь-декабрь 2018
    ("house_3", "2018-09-13 00:00:00", "2018-12-31 23:30:00", +364),
    ("house_3", "2018-08-02 00:00:00", "2018-08-05 23:30:00", +364),
    # дом 4 - пропуск май 2018
    ("house_4", "2018-05-01 03:30:00", "2018-05-14 21:00:00", +364),
    # дом 5 - сбой октябрь-декабрь 2019
    ("house_5", "2019-10-15 00:00:00", "2019-12-06 00:00:00", -364),
    # дом 6 - два блока
    ("house_6", "2017-09-23 03:00:00", "2017-09-27 21:00:00", +364),
    ("house_6", "2017-09-28 00:00:00", "2018-01-07 23:30:00", +364),
    ("house_6", "2018-08-15 04:30:00", "2018-09-26 21:00:00", +364),
    # дом 7 - пропуск март-апрель 2018
    ("house_7", "2018-03-07 06:30:00", "2018-04-28 06:30:00", +364),
]

for col, gap_start, gap_end, days in fill_gaps:
    mask_gap = (
        (df["timestamp"] >= gap_start) &
        (df["timestamp"] <= gap_end)
    )
    n = mask_gap.sum()

    for idx in df[mask_gap].index:
        ts = df.loc[idx, "timestamp"]
        ts_ref = ts + pd.Timedelta(days=days)
        match = df[df["timestamp"] == ts_ref][col]
        if len(match) > 0:
            df.loc[idx, col] = match.values[0]

# короткие пропуски - линейная интерполяция до 24 часов
houses = ["house_1", "house_2", "house_3", "house_4",
          "house_5", "house_6", "house_7", "house_8"]

for col in houses:
    df[col] = df[col].interpolate(method="linear", limit=48, limit_direction="both")

In [ ]:
print("Пропуски после чистки:")
for col in houses:
    n = df[col].isna().sum()
    print(f"{col}: {n}")

In [ ]:
print("Действительные периоды по домам:")
for col in houses:
    first = df[df[col].notna()]["timestamp"].min()
    last  = df[df[col].notna()]["timestamp"].max()
    print(f"{col}: {first} - {last}")

In [ ]:
# обрезаем до общего периода
common_start = "2017-11-11 00:30:00"
common_end = "2019-12-06 10:00:00"

df_final = df[
    (df["timestamp"] >= common_start) &
    (df["timestamp"] <= common_end)
].reset_index(drop=True)

print(f"Период: {df_final['timestamp'].min()} - {df_final['timestamp'].max()}")
print(f"Строк: {len(df_final)}")

In [ ]:
print("Пропуски после обрезки по общему периоду:")
for col in houses:
    n = df_final[col].isna().sum()
    print(f"{col}: {n}")

In [ ]:
# Финальный график сезонности по исходным данным после обработки
window = 7 * 48

fig = go.Figure()

colors = {
    "house_1": "#1f77b4", "house_2": "#ff7f0e", "house_3": "#2ca02c",
    "house_4": "#d62728", "house_5": "#9467bd", "house_6": "#8c564b",
    "house_7": "#e377c2", "house_8": "#7f7f7f"
}

for col in houses:
    rolling = df_final[col].rolling(window=window, center=True).mean()
    fig.add_trace(go.Scatter(
        x=df_final["timestamp"],
        y=rolling,
        mode="lines",
        name=col,
        line=dict(width=1.5, color=colors[col]),
    ))

fig.update_layout(
    title="Скользящее среднее 7 дней по обработанным данным",
    xaxis_title="Дата",
    yaxis_title="Мощность, кВт",
    width=700,
    height=350,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    hovermode="x unified",
    legend=dict(font=dict(size=9))
)

fig.show()
fig.write_image("06_seasonality_clean.png", scale=2)

In [ ]:
#Финальная проверка пропусков
for col in houses:
    n = df_final[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} пропусков - заполняем")
        df_final[col] = df_final[col].interpolate(method="linear")
    else:
        print(f"{col}: 0 пропусков")

In [ ]:
df_final.to_csv("df_final.csv", index=False)